In [2]:
import pymongo as pm 
client = pm.MongoClient( 
'bigdatadb.polito.it', 
ssl=True, 
authSource='carsharing', 
username='ictts', 
password='Ict4SM22!', 
tlsAllowInvalidCertificates=True 
)

In [4]:
db = client['carsharing'] #Choose the DB to use

In [8]:
db.list_collection_names()

ServerSelectionTimeoutError: bigdatadb.polito.it:27017: timed out (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6ad6051bd5c7ebf2427c6bc8, topology_type: Unknown, servers: [<ServerDescription ('bigdatadb.polito.it', 27017) server_type: Unknown, rtt: None, error=NetworkTimeout('bigdatadb.polito.it:27017: timed out (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

In [4]:
#we should analyze Milano, Miami and minchen as group 18

In [9]:
collections = ["ActiveBookings","ActiveParkings","PermanentBookings","PermanentParkings","enjoy_ActiveBookings","enjoy_ActiveParkings","enjoy_PermanentBookings","enjoy_PermanentParkings"]

In [10]:
#How many documents are present in each collection? 
for collection in collections:
    count = db[collection].count_documents({})
    print(collection + ": " + str(count))

KeyboardInterrupt: 

In [8]:
#Why the number of documents in PermanentParkings and PermanentBooking is similar?

In [9]:
#For which cities the system is collecting data?
cityset = set()

for collection in collections:
    city_c = db[collection].distinct("city")
    for one in city_c:
        cityset.add(one)
    
print(cityset) 

{'New York City', 'Hamburg', 'Seattle', 'Milano', 'Vancouver', 'Frankfurt', 'Rheinland', 'Munchen', 'Torino', 'Wien', 'Roma', 'Firenze', 'Denver', 'Austin', 'Washington DC', 'San Diego', 'Toronto', 'Portland', 'Stuttgart', 'Calgary', 'Berlin', 'Madrid', 'Twin Cities', 'Amsterdam', 'Columbus', 'Montreal'}


In [ ]:
#When did each collection start? When did each collection end? 
import pandas as pd
for collection in collections:
    init=db[collection].find({},{"init_date":1,"_id": 0}).sort("init_date",1).limit(1)
    if "Permanent" in collection:
        fin = db[collection].find({}, {"final_date": 1, "_id": 0}).sort("final_date", -1).limit(1)
    else:
        fin = db[collection].find({}, {"init_date": 1, "_id": 0}).sort("init_date", -1).limit(1)
    df_init = pd.DataFrame(list(init))
    df_fin = pd.DataFrame(list(fin))
    print(f"collection:{collection} init date:{df_init.iloc[0,0]} final date:{df_fin.iloc[0,0]}")

In [ ]:
#What about the time zone of the init_date and init_time timestamps? 
#Which time zone do they refer to?

for collection in collections:
    doc = db[collection].find_one()

    if doc is None:
        print(f"{collection}: collezione vuota")
        continue

    if "init_date" not in doc:
        print(f"{collection}: campo init_date non presente")
        continue

    print(f"{collection}: {doc['init_date']}")

#Davide's version
import datetime

pb_coll=db["PermanentBookings"]
res=pb_coll.find_one()

init_converted = datetime.datetime.fromtimestamp(res["init_time"])


print(f"Local date (init_date): {res["init_date"]}") #Local Time
print(f"Converted Timestamp (init_time): {init_converted}") #Coordinated Universal Time (UTC).

    

NameError: name 'collections' is not defined

In [ ]:
#What is the total number of cars seen in the whole period in each city? 
cittaList = ["Milano", "Montreal", "Munchen"]

# un set per ogni città
cittaSet = [set(), set(), set()]

for i in range(3):

    for collection in collections:

        # targhe uniche nella singola collection
        plates = db[collection].distinct("plate", {"city": cittaList[i]})

        # aggiungo tutte le targhe trovate al set della città
        cittaSet[i].update(plates)

    # conteggio finale delle targhe uniche della città
    print(f"The number of cars in {cittaList[i]} is {len(cittaSet[i])}")

#prima abbiamo considerato entrambe i database e poi abbiamo solo caonsiderato car2go,
#ricordando che enjoy è solo in ITalia


The number of cars in Milano is 1155
The number of cars in Montreal is 844
The number of cars in Munchen is 586


In [13]:
#How can you estimate the fleet size in a given period, e.g., one week?
import datetime

cittaList = ["Milano", "Montreal", "Munchen"]

# un set per ogni città
cittaSet = [set(), set(), set()]

# intervallo temporale
start = datetime.datetime(2017, 12, 12, 9, 38, 23)
end   = datetime.datetime(2017, 12, 19, 9, 38, 23)

for i in range(3):
    filtro = {
        "$and": [
            {"city": cittaList[i]},
            {
                "$and": [
                    {"init_date": {"$gte": start}},
                    {"init_date": {"$lte": end}}
                ]
            },
            {
                "$and": [
                    {"final_date": {"$gte": start}},
                    {"final_date": {"$lte": end}}
                ]
            }
        ]
    }

    for collection in collections:

        # targhe uniche nella collection per quella città e periodo
        plates = db[collection].distinct("plate", filtro)

        # aggiungo le targhe al set della città
        cittaSet[i].update(plates)

    print(
        f"The number of cars seen in {cittaList[i]} "
        f"is {len(cittaSet[i])} from {start} to {end}"
    )
    

The number of cars seen in Milano is 802 from 2017-12-12 09:38:23 to 2017-12-19 09:38:23
The number of cars seen in Montreal is 430 from 2017-12-12 09:38:23 to 2017-12-19 09:38:23
The number of cars seen in Munchen is 555 from 2017-12-12 09:38:23 to 2017-12-19 09:38:23


In [15]:
#prima troviamo il range di queste date
import datetime

cittaList = ["Milano", "Montreal", "Munchen"]
cittaIniz = []
cittaFin = []

for city in cittaList:

    min_date = datetime.datetime.max
    max_date = datetime.datetime.min

    for collection in collections:

        # data minima della collection per quella città
        first = db[collection].find({"city": city}).sort("init_date", 1).limit(1)

        # data massima della collection per quella città
        last = db[collection].find(
            {"city": city}
        ).sort("init_date", -1).limit(1)

        first = list(first)
        last = list(last)

        if first:
            if first[0]["init_date"] < min_date:
                min_date = first[0]["init_date"]

        if last:
            if last[0]["init_date"] > max_date:
                max_date = last[0]["init_date"]

    cittaIniz.append(min_date)
    cittaFin.append(max_date)
    print(city)
    print("Min:", min_date)
    print("Max:", max_date)
    print()

    

Milano
Min: 2016-12-13 18:37:38
Max: 2018-01-31 14:11:33

Montreal
Min: 2016-12-13 12:37:38
Max: 2018-01-31 08:11:33

Munchen
Min: 2016-12-13 18:37:38
Max: 2018-01-31 14:10:45



In [ ]:
#How does this relate to the total number of vehicles seen in the whole collection? 
from datetime import datetime, timedelta

cities = ["Milano", "Montreal", "Munchen"]

date_ranges = {
    "Milano": (
        datetime(2016, 12, 13, 18, 37, 38),
        datetime(2018, 1, 31, 14, 11, 33)
    ),
    "Montreal": (
        datetime(2016, 12, 13, 12, 37, 38),
        datetime(2018, 1, 31, 8, 11, 33)
    ),
    "Munchen": (
        datetime(2016, 12, 13, 18, 37, 38),
        datetime(2018, 1, 31, 14, 10, 45)
    )
}

for city in cities:

    print(f"\n===== {city} =====")

    start_date, end_date = date_ranges[city]

    current_start = start_date
    week_number = 1

    while current_start < end_date:

        current_end = current_start + timedelta(days=7)

        plates = set()

        for collection in collections:

            cursor = db[collection].find({
                "city": city,
                "init_date": {
                    "$gte": current_start,
                    "$lt": current_end
                }
            })

            for doc in cursor:
                if "plate" in doc:
                    plates.add(doc["plate"])

        print(
            f"Settimana {week_number}: "
            f"{len(plates)} veicoli unici"
        )

        week_number += 1
        current_start = current_end

In [ ]:
#How many bookings have been recorded in December 2017 in each city? 
import datetime

cittaList = ["Milano", "Montreal", "Munchen"]

# un set per ogni città
cittaSet = [set(), set(), set()]

# intervallo temporale
start = datetime.datetime(2017, 12, 1, 0, 0, 0)
end   = datetime.datetime(2017, 12, 31, 23, 59, 59)

for i in range(3):
    filtro = {
        "$and": [
            {"city": cittaList[i]},
            {
                "$and": [
                    {"init_date": {"$gte": start}},
                    {"init_date": {"$lte": end}}
                ]
            },
            {
                "$and": [
                    {"final_date": {"$gte": start}},
                    {"final_date": {"$lte": end}}
                ]
            }
        ]
    }

    for collection in collections:

        # targhe uniche nella collection per quella città e periodo
        plates = db[collection].distinct("plate", filtro)

        # aggiungo le targhe al set della città
        cittaSet[i].update(plates)

    print(
        f"The number of cars seen in {cittaList[i]} "
        f"is {len(cittaSet[i])} from {start} to {end}"
    )


    #DAVIDE's version
    start=datetime.datetime(2017, 12, 1, 0, 0, 0)
end=datetime.datetime(2017, 12, 31, 23, 59, 59)
PBs=["PermanentBookings","enjoy_PermanentBookings"]

for city in cities:
    total_bookings=0
    for book in PBs:
        bookings=db[book].count_documents(
            {
            "city": city,
                "init_date":{
                    "$gte": start,
                    "$lte": end},
            })
        total_bookings+=bookings
    print(f"the number of bookings in {city} from {start} to {end} is: {total_bookings}")

The number of cars seen in Milano is 818 from 2017-12-01 00:00:00 to 2017-12-31 23:59:59
The number of cars seen in Montreal is 443 from 2017-12-01 00:00:00 to 2017-12-31 23:59:59
The number of cars seen in Munchen is 558 from 2017-12-01 00:00:00 to 2017-12-31 23:59:59


In [ ]:
#How many bookings have the alternative transportation modes recorded in each city? 

for city in cities:
    total_trans=0
    for book in PBs:
        bookings=db[book].count_documents(
            {
            "city": city,
            "public_transport.duration":{"$ne":-1}
            })
        total_trans+=bookings
    print(f"the number of bookings in {city} with alternative transportation modes is: {total_trans}")


In [41]:
db["PermanentBookings"].find_one()

{'_id': ObjectId('5900e8bc2ad8531539ed0847'),
 'init_fuel': 45,
 'city': 'Vancouver',
 'walking': {'duration': -1, 'distance': -1},
 'vendor': 'car2go',
 'driving': {'duration': -1, 'distance': -1},
 'final_time': 1481650748,
 'plate': '644XNS',
 'engineType': 'CE',
 'init_time': 1481650703,
 'vin': 'WMEEJ3BA8EK762841',
 'smartPhoneRequired': False,
 'interior': 'GOOD',
 'final_fuel': 45,
 'exterior': 'GOOD',
 'init_date': datetime.datetime(2016, 12, 13, 9, 38, 23),
 'final_date': datetime.datetime(2016, 12, 13, 9, 39, 8),
 'init_address': '1694 Duranleau,surface lot next t/Bridges.fits3',
 'final_address': '1694 Duranleau,surface lot next t/Bridges.fits3',
 'origin_destination': {'type': 'LineString',
  'coordinates': [[-123.13606, 49.27269], [-123.13606, 49.27269]]},
 'public_transport': {'duration': -1,
  'distance': -1,
  'arrival_date': -1,
  'arrival_time': -1}}

In [ ]:
#How many bookings have the alternative transportation modes recorded in each city?


for city in cities:
    total_trans=0
    for book in PBs:
        bookings=db[book].count_documents(
            {
            "city": city,
            "public_transport.duration":{"$ne":-1}
            })
        total_trans+=bookings
    print(f"the number of bookings in {city} with alternative transportation modes is: {total_trans}")

        

In [ ]:
#1) Derive the distribution of booking/parking duration and plot them. Show the distributions as
#empirical Cumulative Distribution Functions (CDF). Which consideration can you derive from the
#results?
import datetime
import pandas as pd
import matplotlib.pyplot as plt


start = datetime.datetime(2017, 11, 1, 0, 0, 0)
end = datetime.datetime(2018, 1, 31, 23, 59, 59)

cities = ["Milano", "Montreal", "Munchen"] 
PBs = ["PermanentBookings", "enjoy_PermanentBookings"]
PPs = ["PermanentParkings", "enjoy_PermanentParkings"]

match_stage = {
    "$match": {
        "city": {"$in": cities}, # Extracts all 3 cities in a single query
        "init_date": {"$gte": start, "$lte": end}
    }
}

group_stage = {
    "$group": {
        # Group simultaneously by city and duration
        "_id": {
            "city": "$city",
            "duration_min": {
                "$floor": {
                    "$divide": [{"$subtract": ["$final_time", "$init_time"]}, 60]
                }
            }
        },
        "count": {"$sum": 1}
    }
}

# 2. Download the data making only 4 total queries to the database
book_results = []
for book in PBs:
    cursor = db[book].aggregate([match_stage, group_stage])
    book_results.extend(list(cursor))

park_results = []
for park in PPs:
    cursor = db[park].aggregate([match_stage, group_stage])
    park_results.extend(list(cursor))

# 3. Transform into a DataFrame and "unpack" the composite id
df_book_all = pd.DataFrame()
if book_results:
    df_temp = pd.DataFrame(book_results)
    # Extract city and duration from the '_id' dictionary
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['duration_min'] = df_temp['_id'].apply(lambda x: x['duration_min'])
    # Group by to merge the counts of Enjoy and Car2Go
    df_book_all = df_temp.groupby(['city', 'duration_min'])['count'].sum().reset_index()

df_park_all = pd.DataFrame()
if park_results:
    df_temp = pd.DataFrame(park_results)
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['duration_min'] = df_temp['_id'].apply(lambda x: x['duration_min'])
    df_park_all = df_temp.groupby(['city', 'duration_min'])['count'].sum().reset_index()

# 4. Generate the plots by filtering in Python (instant operation in RAM)
for i in cities:
    plt.figure(figsize=(10, 6))
    
    # Filter the global DataFrame for the current city
    if not df_book_all.empty:
        df_b = df_book_all[df_book_all['city'] == i].sort_values('duration_min').copy()
        if not df_b.empty:
            df_b['cdf'] = df_b['count'].cumsum() / df_b['count'].sum()
            plt.plot(df_b['duration_min'], df_b['cdf'], label='Bookings', color='blue')
            
    if not df_park_all.empty:
        df_p = df_park_all[df_park_all['city'] == i].sort_values('duration_min').copy()
        if not df_p.empty:
            df_p['cdf'] = df_p['count'].cumsum() / df_p['count'].sum()
            plt.plot(df_p['duration_min'], df_p['cdf'], label='Parkings', color='red')

    plt.xlim(0, 300)
    plt.title(f"CDF of Booking vs Parking Duration - {i} (Nov 17 - Jan 18)")
    plt.xlabel("Duration (minutes)")
    plt.ylabel("Cumulative Probability")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots side-by-side (1 row, 2 columns)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = ['blue', 'green', 'orange']

# --- PLOT 1: BOOKINGS ---
for idx, city in enumerate(cities):
    if not df_book_all.empty:
        df_b = df_book_all[df_book_all['city'] == city].sort_values('duration_min').copy()
        if not df_b.empty:
            df_b['cdf'] = df_b['count'].cumsum() / df_b['count'].sum()
            ax1.plot(df_b['duration_min'], df_b['cdf'], label=city, color=colors[idx])

ax1.set_xlim(0, 150) # Limit to 150 minutes for bookings
ax1.set_title("Bookings Comparison Across Cities")
ax1.set_xlabel("Duration (minutes)")
ax1.set_ylabel("Cumulative Probability")
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# --- PLOT 2: PARKINGS ---
for idx, city in enumerate(cities):
    if not df_park_all.empty:
        df_p = df_park_all[df_park_all['city'] == city].sort_values('duration_min').copy()
        if not df_p.empty:
            df_p['cdf'] = df_p['count'].cumsum() / df_p['count'].sum()
            ax2.plot(df_p['duration_min'], df_p['cdf'], label=city, color=colors[idx])

ax2.set_xlim(0, 500) # Wider limit for parkings (e.g., overnight)
ax2.set_title("Parkings Comparison Across Cities")
ax2.set_xlabel("Duration (minutes)")
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
#Does the CDF change over time? E.g., aggregate per different weeks of data, or per different
#days. Are these CDFs different? Why?
import datetime
import pandas as pd
import matplotlib.pyplot as plt
start = datetime.datetime(2017, 11, 1, 0, 0, 0)
end = datetime.datetime(2018, 1, 31, 23, 59, 59)

cities = ["Milano", "Montreal", "Munchen"] 
PBs = ["PermanentBookings", "enjoy_PermanentBookings"]
PPs = ["PermanentParkings", "enjoy_PermanentParkings"] 
match_stage = {
    "$match": {
        "city": {"$in": cities}, # Extracts all 3 cities in a single query
        "init_date": {"$gte": start, "$lte": end}
    }
}

group_stage = {
    "$group": {
        "_id": {
            "city": "$city",
            "day_of_week": {"$dayOfWeek": "$init_date"},
            "duration_min": {
                "$floor": {
                    "$divide": [{"$subtract": ["$final_time", "$init_time"]}, 60]
                }
            }
        },
        "count": {"$sum": 1}
    }
}
book_results = []
for book in PBs:
    cursor = db[book].aggregate([match_stage, group_stage])
    book_results.extend(list(cursor))

park_results = []
for park in PPs:
    cursor = db[park].aggregate([match_stage, group_stage])
    park_results.extend(list(cursor))

df_book_all = pd.DataFrame()
if book_results:
    df_temp = pd.DataFrame(book_results)
    # Extract city and duration from the '_id' dictionary
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['duration_min'] = df_temp['_id'].apply(lambda x: x['duration_min'])
    df_temp['day_of_week']=df_temp['_id'].apply(lambda x: x['day_of_week'])
    # Group by to merge the counts of Enjoy and Car2Go
    df_book_all = df_temp.groupby(['city','day_of_week', 'duration_min'])['count'].sum().reset_index()

df_park_all = pd.DataFrame()
if park_results:
    df_temp = pd.DataFrame(park_results)
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['duration_min'] = df_temp['_id'].apply(lambda x: x['duration_min'])
    df_temp['day_of_week']=df_temp['_id'].apply(lambda x: x['day_of_week'])
    df_park_all = df_temp.groupby(['city','day_of_week', 'duration_min'])['count'].sum().reset_index()

weekend_days = [1, 7] 

# Create a figure with 3 subplots side-by-side (1 row, 3 columns)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, city in enumerate(cities):
    ax = axes[idx]
    
    if not df_book_all.empty:
        # Filter data for the current city
        df_city = df_book_all[df_book_all['city'] == city]
        
        # Separate Weekends and Weekdays
        df_weekend = df_city[df_city['day_of_week'].isin(weekend_days)].copy()
        df_weekday = df_city[~df_city['day_of_week'].isin(weekend_days)].copy()
        
        # Regroup by duration, summing the counts across different days
        df_weekend = df_weekend.groupby('duration_min')['count'].sum().reset_index().sort_values('duration_min')
        df_weekday = df_weekday.groupby('duration_min')['count'].sum().reset_index().sort_values('duration_min')
        
        # Calculate and plot the CDF for Weekdays
        if not df_weekday.empty:
            df_weekday['cdf'] = df_weekday['count'].cumsum() / df_weekday['count'].sum()
            ax.plot(df_weekday['duration_min'], df_weekday['cdf'], label='Weekday (Mon-Fri)', color='blue')
            
        # Calculate and plot the CDF for Weekends
        if not df_weekend.empty:
            df_weekend['cdf'] = df_weekend['count'].cumsum() / df_weekend['count'].sum()
            ax.plot(df_weekend['duration_min'], df_weekend['cdf'], label='Weekend (Sat-Sun)', color='red')
            
    # Settings for each individual subplot
    ax.set_xlim(0, 150) # Limit to 150 minutes to highlight the difference
    ax.set_title(f"Bookings: Weekday vs Weekend - {city}")
    ax.set_xlabel("Duration (minutes)")
    if idx == 0:
        ax.set_ylabel("Cumulative Probability")
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
#Consider the system utilization over time: aggregate rentals per hour of the day, and then plot the
#number of booked/parked cars (or percentage of booked/parked cars) per hour versus time of day.
#Do you notice any outliers? Can you explain them?

import datetime
import pandas as pd
import matplotlib.pyplot as plt

start = datetime.datetime(2017, 11, 1, 0, 0, 0)
end = datetime.datetime(2018, 1, 31, 23, 59, 59)

cities = ["Milano", "Montreal", "Munchen"] 
PBs = ["PermanentBookings", "enjoy_PermanentBookings"]
PPs = ["PermanentParkings", "enjoy_PermanentParkings"] 
match_stage = {
    "$match": {
        "city": {"$in": cities}, # Extracts all 3 cities in a single query
        "init_date": {"$gte": start, "$lte": end}
    }
}

group_stage = {
    "$group": {
        "_id": {
            "city": "$city",
            "hour": {"$hour": "$init_date"}
        },
        "count": {"$sum": 1}
    }
}
book_results = []
for book in PBs:
    cursor = db[book].aggregate([match_stage, group_stage])
    book_results.extend(list(cursor))

park_results = []
for park in PPs:
    cursor = db[park].aggregate([match_stage, group_stage])
    park_results.extend(list(cursor))

df_book_all = pd.DataFrame()
if book_results:
    df_temp = pd.DataFrame(book_results)
    # Extract city and duration from the '_id' dictionary
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['hour']=df_temp['_id'].apply(lambda x: x['hour'])
    # Group by to merge the counts of Enjoy and Car2Go
    df_book_all = df_temp.groupby(['city','hour'])['count'].sum().reset_index()

df_park_all = pd.DataFrame()
if park_results:
    df_temp = pd.DataFrame(park_results)
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['hour']=df_temp['_id'].apply(lambda x: x['hour'])
    df_park_all = df_temp.groupby(['city','hour'])['count'].sum().reset_index()

def plot_hourly_trend(df, title):
    
    if df.empty:
        print(f"no data for {title}")
        return
        
    plt.figure(figsize=(10, 6))
    colors = ['blue', 'green', 'orange']
    cities = df['city'].unique()
    
    for idx, city in enumerate(cities):
        
        df_city = df[df['city'] == city].sort_values('hour').copy()
        
        
        total_events = df_city['count'].sum()
        df_city['percentage'] = (df_city['count'] / total_events) * 100
        
        
        plt.plot(df_city['hour'], df_city['percentage'], label=city, color=colors[idx], marker='o')

   
    plt.title(title)
    plt.xlabel("Hour of the Day (0-23)")
    plt.ylabel("Percentage of Total Events (%)")
    plt.xticks(range(0, 24)) 
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

plot_hourly_trend(df_book_all, "System Utilization: Bookings per Hour")
plot_hourly_trend(df_park_all, "System Utilization: Parkings per Hour")

In [ ]:
total_book_unfiltered = df_book_all['count'].sum()
total_park_unfiltered = df_park_all['count'].sum()


print(f"Total bookings (UNFILTERED): {total_book_unfiltered}")
print(f"Total Parkings (UNFILTERED): {total_park_unfiltered}")

In [ ]:
#Derive a criterion to filter possible outliers (e.g., booking periods that are too short/too long), to
#obtain rentals from bookings, filtering system issues or problems with the data collection.
lower_bound_book=5
upper_bound_book=1440 #24h
lower_bound_park=0
upper_bound_park=4320 #3day
match_stage = {
    "$match": {
        "city": {"$in": cities}, # Extracts all 3 cities in a single query
        "init_date": {"$gte": start, "$lte": end}
    }
}
project_stage= {
    "$project":{
        "city":1,
        "init_date":1,
        "duration_min": {
            "$floor": {
                "$divide": [{"$subtract": ["$final_time", "$init_time"]}, 60]
            }
        }
    }
}
match_stage2_book={
    "$match":{
        "duration_min":{"$gte":lower_bound_book,"$lte":upper_bound_book}
    }
}
match_stage2_park={
    "$match":{
        "duration_min":{"$gte":lower_bound_park,"$lte":upper_bound_park}
    }
}

group_stage = {
    "$group": {
        "_id": {
            "city": "$city",
            "hour": {"$hour": "$init_date"}
        },
        "count": {"$sum": 1}
    }
}
book_results = []
for book in PBs:
    cursor = db[book].aggregate([match_stage,project_stage,match_stage2_book, group_stage])
    book_results.extend(list(cursor))

park_results = []
for park in PPs:
    cursor = db[park].aggregate([match_stage,project_stage,match_stage2_park, group_stage])
    park_results.extend(list(cursor))

df_book_all = pd.DataFrame()
if book_results:
    df_temp = pd.DataFrame(book_results)
    # Extract city and duration from the '_id' dictionary
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['hour']=df_temp['_id'].apply(lambda x: x['hour'])
    # Group by to merge the counts of Enjoy and Car2Go
    df_book_all = df_temp.groupby(['city','hour'])['count'].sum().reset_index()

df_park_all = pd.DataFrame()
if park_results:
    df_temp = pd.DataFrame(park_results)
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['hour']=df_temp['_id'].apply(lambda x: x['hour'])
    df_park_all = df_temp.groupby(['city','hour'])['count'].sum().reset_index()

plot_hourly_trend(df_book_all, "System Utilization: Bookings per Hour-FILTERED")
plot_hourly_trend(df_park_all, "System Utilization: Parkings per Hour-FILTERED")

In [ ]:
total_book_filtered = df_book_all['count'].sum()
total_park_filtered = df_park_all['count'].sum()


print(f"Total bookings (FILTERED): {total_book_filtered}")
print(f"Total Parkings (FILTERED): {total_park_filtered}")

In [ ]:
group_stage = {
    "$group": {
        "_id": {
            "city": "$city",
            "duration_min": "$duration_min" 
        },
        "count": {"$sum": 1}
    }
}
book_results = []
for book in PBs:
    cursor = db[book].aggregate([match_stage,project_stage,match_stage2_book,group_stage])
    book_results.extend(list(cursor))

park_results = []
for park in PPs:
    cursor = db[park].aggregate([match_stage,project_stage,match_stage2_park,group_stage])
    park_results.extend(list(cursor))

# 3. Transform into a DataFrame and "unpack" the composite id
df_book_all_fil = pd.DataFrame()
if book_results:
    df_temp = pd.DataFrame(book_results)
    # Extract city and duration from the '_id' dictionary
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['duration_min'] = df_temp['_id'].apply(lambda x: x['duration_min'])
    # Group by to merge the counts of Enjoy and Car2Go
    df_book_all_fil = df_temp.groupby(['city', 'duration_min'])['count'].sum().reset_index()

df_park_all_fil = pd.DataFrame()
if park_results:
    df_temp = pd.DataFrame(park_results)
    df_temp['city'] = df_temp['_id'].apply(lambda x: x['city'])
    df_temp['duration_min'] = df_temp['_id'].apply(lambda x: x['duration_min'])
    df_park_all_fil = df_temp.groupby(['city', 'duration_min'])['count'].sum().reset_index()


plot_cdf_booking_vs_parking(df_book_all_fil, df_park_all_fil, cities, x_limit=300, title_suffix="(filtered)")

In [ ]:
group_stage_daily = {
    "$group": {
        "_id": {
            "city": "$city",
            "day": {"$dateToString": {"format": "%Y-%m-%d", "date": "$init_date"}},
            "duration_min": "$duration_min" 
        },
        "count": {"$sum": 1}
    }
}
book_results = []
for book in PBs:
    cursor = db[book].aggregate([match_stage, project_stage, match_stage2_book, group_stage_daily])
    book_results.extend(list(cursor))

park_results = []
for park in PPs:
    cursor = db[park].aggregate([match_stage, project_stage, match_stage2_park, group_stage_daily])
    park_results.extend(list(cursor))

# 3. Funzione per estrarre e calcolare le metriche statistiche giornaliere
def calculate_daily_stats(results_list):
    if not results_list:
        return pd.DataFrame()
        
    df = pd.DataFrame(results_list)
    df['city'] = df['_id'].apply(lambda x: x['city'])
    df['day'] = df['_id'].apply(lambda x: x['day'])
    df['duration_min'] = df['_id'].apply(lambda x: x['duration_min'])
    
    df_grouped = df.groupby(['city', 'day', 'duration_min'])['count'].sum().reset_index()
    

    df_expanded = df_grouped.loc[df_grouped.index.repeat(df_grouped['count'])]
    
    stats = df_expanded.groupby(['city', 'day'])['duration_min'].agg(
        average='mean',
        median='median',
        std_dev='std',
        percentile_25=lambda x: x.quantile(0.25),
        percentile_75=lambda x: x.quantile(0.75),
        percentile_90=lambda x: x.quantile(0.90)
    ).reset_index()
    

    stats['day'] = pd.to_datetime(stats['day'])
    return stats.sort_values(['city', 'day'])


df_book_stats = calculate_daily_stats(book_results)
df_park_stats = calculate_daily_stats(park_results)


def plot_daily_metrics(df_stats, title):
    if df_stats.empty:
        return
        
    cities = df_stats['city'].unique()
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    
    for idx, city in enumerate(cities):
        ax = axes[idx]
        df_city = df_stats[df_stats['city'] == city]
        
        ax.plot(df_city['day'], df_city['average'], label='Average (Mean)', color='blue', linewidth=2)
        ax.plot(df_city['day'], df_city['median'], label='Median', color='green', linewidth=2)
        
        ax.fill_between(df_city['day'], df_city['percentile_25'], df_city['percentile_75'], 
                        color='gray', alpha=0.3, label='25th-75th Percentile')
        
        ax.set_title(f"{title} - {city}")
        ax.set_ylabel("Duration (minutes)")
        ax.legend(loc="upper right")
        ax.grid(True, linestyle='--', alpha=0.6)

    plt.xlabel("Date (Nov 2017 - Jan 2018)")
    plt.tight_layout()
    plt.show()

plot_daily_metrics(df_book_stats, "Daily Statistics for Filtered Bookings")
plot_daily_metrics(df_park_stats, "Daily Statistics for Filtered Parkings")